In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
  from google.colab import files
uploaded = files.upload()



Saving youcookii_annotations_trainval.json to youcookii_annotations_trainval.json


In [ ]:
!pip install yt-dlp


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 54.2 MB/s eta 0:00:00


In [ ]:
from google.colab import files
cookies_file = files.upload()


Saving youtube.com_cookies.txt to youtube.com_cookies.txt


In [4]:
import os

full_dir = "/content/drive/MyDrive/youcook2_full_videos"   # already downloaded videos
clip_dir = "/content/drive/MyDrive/youcook2_clips"         # final clips
processed_file = "/content/drive/MyDrive/processed_videos.txt"

os.makedirs(clip_dir, exist_ok=True)

# Create processed file if doesn't exist
if not os.path.exists(processed_file):
    with open(processed_file, "w") as f:
        pass


In [5]:
import json

with open("youcookii_annotations_trainval.json") as f:
    ann = json.load(f)

with open(processed_file) as f:
    processed_ids = set(line.strip() for line in f)

print("Already processed:", len(processed_ids))


Already processed: 318


In [6]:
import os

clip_dir = "/content/drive/MyDrive/youcook2_clips"
num_files = len(os.listdir(clip_dir))
print(f"Number of files in {clip_dir}: {num_files}")

Number of files in /content/drive/MyDrive/youcook2_clips: 2471


In [7]:
import os

# Folder where your existing clips are
src_dir = "/content/drive/MyDrive/youcook2_clips"

# Folder where compressed clips will be saved
dst_dir = "/content/drive/MyDrive/youcook2_clips_small"

os.makedirs(dst_dir, exist_ok=True)

print("Source clips:", len(os.listdir(src_dir)))
print("Destination folder ready.")


Source clips: 2471
Destination folder ready.


In [12]:
import json
import re

ANNOT_FILE = "youcookii_annotations_trainval.json"

# Load annotations (new format with "database")
with open(ANNOT_FILE, "r") as f:
    YC_FULL = json.load(f)

YC = YC_FULL["database"]  # <-- IMPORTANT FIX

import json
import re

ANNOT_FILE = "youcookii_annotations_trainval.json"

# Load annotations (new format with "database")
with open(ANNOT_FILE, "r") as f:
    YC_FULL = json.load(f)

YC = YC_FULL["database"]  # corrected JSON root




def parse_filename(fname):
    """Fixed version handling video IDs with underscores."""
    fname = fname.replace(".mp4", "")
    parts = fname.split("_")

    if len(parts) < 4:
        return None, None, None

    try:
        end = int(parts[-1])
        start = int(parts[-2])
        video_id = "_".join(parts[:-3])  # All parts except last 3
        return video_id, start, end
    except (ValueError, IndexError):
        return None, None, None


def get_gt_caption(filename):
    """Return caption ONLY if exact segment match exists in JSON."""
    video_id, start, end = parse_filename(filename)

    if video_id is None or video_id not in YC:
        return None

    # Exact match only (works since JSON has integers)
    for ann in YC[video_id]["annotations"]:
        seg_start, seg_end = ann["segment"]
        if seg_start == start and seg_end == end:
            return ann["sentence"]

    return None


# Test with the problematic clips
print("\nTesting with previously missing clips:")
test_missing = [
    "M4cDslY_qCg_seg0_13_32.mp4",
    "-sQXBqu-_1w_seg0_53_70.mp4"
]

for f in test_missing:
    gt = get_gt_caption(f)
    print(f"{f}: {'FOUND' if gt else 'NOT FOUND'}")
    if gt:
        print(f"  Caption: {gt[:50]}...")



Testing with previously missing clips:
M4cDslY_qCg_seg0_13_32.mp4: FOUND
  Caption: add some olive oil 1 clove garlic salt and pepper ...
-sQXBqu-_1w_seg0_53_70.mp4: FOUND
  Caption: slice the pork...


In [13]:
import tensorflow as tf
import cv2
import numpy as np
import os
import random

CLIPS_DIR = "/content/drive/MyDrive/youcook2_clips_small"
FRAMES_PER_CLIP = 12
IMG_SIZE = 128

def load_video_frames(path):
    cap = cv2.VideoCapture(path)
    frames = []
    while True:
        ret, f = cap.read()
        if not ret:
            break
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        frames.append(f)
    cap.release()

    if len(frames) == 0:
        return np.zeros((FRAMES_PER_CLIP, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)

    # Uniform sampling
    idxs = np.linspace(0, len(frames)-1, FRAMES_PER_CLIP).astype(int)
    sampled = [frames[i] for i in idxs]

    # Resize + normalize
    processed = []
    for f in sampled:
        f = cv2.resize(f, (IMG_SIZE, IMG_SIZE))
        f = f.astype(np.float32) / 255.0
        processed.append(f)

    return np.array(processed)


In [14]:
all_clips = [f for f in os.listdir(CLIPS_DIR) if f.endswith(".mp4")]
print(all_clips[:5])
print("Total clips:", len(all_clips))
dataset_pairs = []
for f in all_clips[0:500]:
    gt = get_gt_caption(f)
    if gt:
        dataset_pairs.append((f, gt))

print("Total usable segments:", len(dataset_pairs))


['pTjoGIvSfE8_seg0_154_181.mp4', 'pTjoGIvSfE8_seg1_221_273.mp4', 'pTjoGIvSfE8_seg2_275_357.mp4', 'pTjoGIvSfE8_seg3_361_390.mp4', 'pTjoGIvSfE8_seg4_396_410.mp4']
Total clips: 2395
Total usable segments: 500


In [15]:
from tensorflow.keras.layers import TextVectorization

captions = [cap for _, cap in dataset_pairs]

vectorizer = TextVectorization(
    max_tokens=5000,
    output_sequence_length=25,
    standardize='lower_and_strip_punctuation'
)

vectorizer.adapt(captions)

VOCAB_SIZE = len(vectorizer.get_vocabulary())
print("Vocab size:", VOCAB_SIZE)


Vocab size: 612


In [30]:
# Test the vectorizer on some examples
test_captions = captions[:5]
vectorized = vectorizer(test_captions)
print("\nVectorization test:")
for i, (cap, vec) in enumerate(zip(test_captions, vectorized.numpy())):
    print(f"\nCaption {i+1}: {cap}")
    print(f"Vectorized: {vec[:15]}...")  # First 15 tokens
    # Count non-zero entries (actual words)
    num_words = sum(1 for x in vec if x != 0)
    print(f"Number of tokens after vectorization: {num_words}")

# Check OOV (out-of-vocabulary) rate
print(f"\nVocabulary size: {VOCAB_SIZE}")
print(f"Vocabulary sample: {vectorizer.get_vocabulary()[:20]}")  # First 20 words


Vectorization test:

Caption 1: drain and rinse cannellini beans and set aside
Vectorized: [ 52   3 311 584 204   3 301 373   0   0   0   0   0   0   0]...
Number of tokens after vectorization: 8

Caption 2: saute minced garlic until translucent
Vectorized: [ 86 101  26  54 401   0   0   0   0   0   0   0   0   0   0]...
Number of tokens after vectorization: 5

Caption 3: add tomatoes salt pepper dried oregano dried basil fresh parsley and fresh basil
Vectorized: [  4  77  13  21 196 323 196 206 120 117   3 120 206   0   0]...
Number of tokens after vectorization: 13

Caption 4: stir over medium heat until mixture comes to a boil
Vectorized: [ 12  49 235  44  54  40 562   5   6  60   0   0   0   0   0]...
Number of tokens after vectorization: 10

Caption 5: reduce heat to low and simmer for 15-20 minutes until seasonings and tomatoes are well blended
Vectorized: [455  44   5 331   3 294  66 378  87  54 444   3  77 601  76]...
Number of tokens after vectorization: 16

Vocabulary size: 

In [17]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# ============================================
# VIDEO CONFIGURATION (Based on your dataset)
# ============================================
FRAMES_PER_CLIP = 15    # Number of frames per video clip (adjust based on your data)
IMG_HEIGHT = 224        # Input image height
IMG_WIDTH = 224         # Input image width
IMG_CHANNELS = 3        # RGB channels

# ============================================
# TEXT/CAPTION CONFIGURATION
# ============================================
# From your previous vectorizer setup:
MAX_CAPTION_LENGTH = 25     # From vectorizer output_sequence_length
VOCAB_SIZE = 612           # From vectorizer.get_vocabulary() length

# ============================================
# MODEL ARCHITECTURE HYPERPARAMETERS
# ============================================
EMBED_DIM = 256         # Embedding dimension (common size, divisible by NUM_HEADS)
NUM_HEADS = 4           # Number of attention heads (EMBED_DIM must be divisible by this)
ENC_LAYERS = 2          # Number of transformer encoder layers
DEC_LAYERS = 2          # Number of transformer decoder layers (for future use)
FF_DIM = 512            # Feed-forward network dimension (typically 2-4x EMBED_DIM)
DROPOUT_RATE = 0.1      # Dropout rate for regularization

# ============================================
# TRAINING HYPERPARAMETERS
# ============================================
BATCH_SIZE = 16         # Batch size (adjust based on GPU memory)
LEARNING_RATE = 1e-4    # Learning rate
EPOCHS = 10            # Number of training epochs

In [18]:
def build_video_encoder():
    """
    Builds a video encoder using CNN + Transformer architecture.

    Input shape:  (batch_size, FRAMES_PER_CLIP, IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)
    Output shape: (batch_size, FRAMES_PER_CLIP, EMBED_DIM)
    """
    # Input layer
    video_input = layers.Input(
        shape=(FRAMES_PER_CLIP, IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS),
        name="video_input"
    )

    # -------------------------------------------------
    # CNN Feature Extraction (per frame)
    # -------------------------------------------------
    # Layer 1: Conv + Pool
    x = layers.TimeDistributed(
        layers.Conv2D(
            filters=32,
            kernel_size=3,
            padding='same',
            activation='relu',
            name="frame_conv1"
        ),
        name="td_conv1"
    )(video_input)
    x = layers.TimeDistributed(
        layers.MaxPooling2D(pool_size=(2, 2), name="frame_pool1"),
        name="td_pool1"
    )(x)  # Output: (batch, frames, 112, 112, 32)

    # Layer 2: Conv + Pool
    x = layers.TimeDistributed(
        layers.Conv2D(
            filters=64,
            kernel_size=3,
            padding='same',
            activation='relu',
            name="frame_conv2"
        ),
        name="td_conv2"
    )(x)
    x = layers.TimeDistributed(
        layers.MaxPooling2D(pool_size=(2, 2), name="frame_pool2"),
        name="td_pool2"
    )(x)  # Output: (batch, frames, 56, 56, 64)

    # Layer 3: Conv + Pool
    x = layers.TimeDistributed(
        layers.Conv2D(
            filters=128,
            kernel_size=3,
            padding='same',
            activation='relu',
            name="frame_conv3"
        ),
        name="td_conv3"
    )(x)
    x = layers.TimeDistributed(
        layers.GlobalAveragePooling2D(name="frame_global_pool"),
        name="td_global_pool"
    )(x)  # Output: (batch, frames, 128)

    # -------------------------------------------------
    # Linear Projection to Embedding Dimension
    # -------------------------------------------------
    x = layers.Dense(
        EMBED_DIM,
        activation='linear',
        name="feature_projection"
    )(x)  # Output: (batch, frames, EMBED_DIM)

    # -------------------------------------------------
    # Positional Encoding
    # -------------------------------------------------
    # Create position indices
    positions = tf.range(start=0, limit=FRAMES_PER_CLIP, delta=1)

    # Position embedding layer
    position_embedding = layers.Embedding(
        input_dim=FRAMES_PER_CLIP,
        output_dim=EMBED_DIM,
        name="position_embedding"
    )

    # Get position embeddings and add batch dimension
    pos_emb = position_embedding(positions)  # Shape: (frames, EMBED_DIM)
    pos_emb = tf.expand_dims(pos_emb, axis=0)  # Shape: (1, frames, EMBED_DIM)

    # Add positional embeddings to features
    x = layers.Add(name="add_positional")([x, pos_emb])

    # Optional: Add dropout for regularization
    x = layers.Dropout(DROPOUT_RATE, name="encoder_dropout_init")(x)

    # -------------------------------------------------
    # Transformer Encoder Layers
    # -------------------------------------------------
    for layer_idx in range(ENC_LAYERS):
        # Multi-Head Self Attention
        attention_output = layers.MultiHeadAttention(
            num_heads=NUM_HEADS,
            key_dim=EMBED_DIM // NUM_HEADS,  # Ensure proper dimensionality
            dropout=DROPOUT_RATE,
            name=f"encoder_mha_{layer_idx}"
        )(x, x)

        # Add & Norm (Residual connection 1)
        x = layers.Add(name=f"encoder_add1_{layer_idx}")([x, attention_output])
        x = layers.LayerNormalization(
            epsilon=1e-6,
            name=f"encoder_norm1_{layer_idx}"
        )(x)

        # Feed-Forward Network
        ff_output = layers.Dense(
            FF_DIM,
            activation='relu',
            name=f"encoder_ff1_{layer_idx}"
        )(x)
        ff_output = layers.Dropout(DROPOUT_RATE)(ff_output)
        ff_output = layers.Dense(
            EMBED_DIM,
            activation='linear',
            name=f"encoder_ff2_{layer_idx}"
        )(ff_output)

        # Add & Norm (Residual connection 2)
        x = layers.Add(name=f"encoder_add2_{layer_idx}")([x, ff_output])
        x = layers.LayerNormalization(
            epsilon=1e-6,
            name=f"encoder_norm2_{layer_idx}"
        )(x)

    # Final encoder output
    encoder_output = x  # Shape: (batch, frames, EMBED_DIM)

    return Model(
        inputs=video_input,
        outputs=encoder_output,
        name="video_encoder"
    )

In [ ]:
# @title
def test_encoder():
    """Test the video encoder with dummy data."""

    # Build the encoder
    encoder = build_video_encoder()

    # Print model summary
    print("=" * 60)
    print("VIDEO ENCODER SUMMARY")
    print("=" * 60)
    encoder.summary()

    # Create dummy input matching your dataset
    batch_size = BATCH_SIZE
    dummy_video = tf.random.normal(shape=(
        batch_size,
        FRAMES_PER_CLIP,
        IMG_HEIGHT,
        IMG_WIDTH,
        IMG_CHANNELS
    ))

    print(f"\nTest Input Shape: {dummy_video.shape}")
    print(f"Expected: (batch={batch_size}, frames={FRAMES_PER_CLIP}, ")
    print(f"          height={IMG_HEIGHT}, width={IMG_WIDTH}, channels={IMG_CHANNELS})")

    # Forward pass
    output = encoder(dummy_video)
    print(f"\nOutput Shape: {output.shape}")
    print(f"Expected: (batch={batch_size}, frames={FRAMES_PER_CLIP}, embed_dim={EMBED_DIM})")

    # Test with different batch sizes
    print("\n" + "=" * 60)
    print("TESTING WITH DIFFERENT BATCH SIZES")
    print("=" * 60)

    for test_batch in [1, 4, 16, 32]:
        test_input = tf.random.normal(shape=(
            test_batch,
            FRAMES_PER_CLIP,
            IMG_HEIGHT,
            IMG_WIDTH,
            IMG_CHANNELS
        ))
        test_output = encoder(test_input)
        print(f"Batch {test_batch:2d}: Input {test_input.shape} -> Output {test_output.shape}")

    return encoder


if __name__ == "__main__":
    # Run tests
    encoder = test_encoder()

    # Save model diagram
    try:
        tf.keras.utils.plot_model(
            encoder,
            to_file="video_encoder_architecture.png",
            show_shapes=True,
            show_layer_names=True,
            expand_nested=True
        )
        print("\n✓ Model architecture saved as 'video_encoder_architecture.png'")
    except:
        print("\nNote: Install graphviz to save model diagram")
        print("pip install graphviz")

VIDEO ENCODER SUMMARY


Model: "video_encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ video_input         │ (None, 30, 224,   │          0 │ -                 │
│ (InputLayer)        │ 224, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ td_conv1            │ (None, 30, 224,   │        896 │ video_input[0][0] │
│ (TimeDistributed)   │ 224, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ td_pool1            │ (None, 30, 112,   │          0 │ td_conv1[0][0]    │
│ (TimeDistributed)   │ 112, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ td_conv2            │ (None, 30, 112,   │     18,496 │ td_pool1[0][0]    │
│ (TimeDistributed)   │ 112, 64)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ td_pool2            │ (None, 30, 56,    │          0 │ td_conv2[0][0]    │
│ (TimeDistributed)   │ 56, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ td_conv3            │ (None, 30, 56,    │     73,856 │ td_pool2[0][0]    │
│ (TimeDistributed)   │ 56, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ td_global_pool      │ (None, 30, 128)   │          0 │ td_conv3[0][0]    │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_projection  │ (None, 30, 256)   │     33,024 │ td_global_pool[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_positional      │ (1, 30, 256)      │          0 │ feature_projecti… │
│ (Add)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_dropout_in… │ (1, 30, 256)      │          0 │ add_positional[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_mha_0       │ (1, 30, 256)      │    263,168 │ encoder_dropout_… │
│ (MultiHeadAttentio… │                   │            │ encoder_dropout_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_add1_0      │ (1, 30, 256)      │          0 │ encoder_dropout_… │
│ (Add)               │                   │            │ encoder_mha_0[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_norm1_0     │ (1, 30, 256)      │        512 │ encoder_add1_0[0… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_ff1_0       │ (1, 30, 512)      │    131,584 │ encoder_norm1_0[… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (1, 30, 512)      │          0 │ encoder_ff1_0[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_ff2_0       │ (1, 30, 256)      │    131,328 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_add2_0      │ (1, 30, 256)      │          0 │ encoder_norm1_0[… │
│ (Add)               │                   │            │ encoder_ff2_0[0]

 Total params: 1,180,480 (4.50 MB)

 Trainable params: 1,180,480 (4.50 MB)

 Non-trainable params: 0 (0.00 B)


Test Input Shape: (16, 30, 224, 224, 3)
Expected: (batch=16, frames=30, 
          height=224, width=224, channels=3)

Output Shape: (16, 30, 256)
Expected: (batch=16, frames=30, embed_dim=256)

TESTING WITH DIFFERENT BATCH SIZES
Batch  1: Input (1, 30, 224, 224, 3) -> Output (1, 30, 256)
Batch  4: Input (4, 30, 224, 224, 3) -> Output (4, 30, 256)
Batch 16: Input (16, 30, 224, 224, 3) -> Output (16, 30, 256)


In [20]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# ============================================
# CONSTANTS (from previous setup)
# ============================================
# Video configuration
FRAMES_PER_CLIP = 15
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_CHANNELS = 3

# Text configuration
MAX_CAPTION_LENGTH = 25
VOCAB_SIZE = 612

# Model architecture
EMBED_DIM = 256
NUM_HEADS = 4
ENC_LAYERS = 2
DEC_LAYERS = 2
FF_DIM = 512
DROPOUT_RATE = 0.1

# Training
BATCH_SIZE = 16
LEARNING_RATE = 1e-4

# ============================================
# TEXT EMBEDDING LAYER (REUSING YOUR VECTORIZER)
# ============================================
def create_text_embedding_layer(vectorizer):
    """
    Creates a text embedding layer from your existing vectorizer.

    Args:
        vectorizer: Your trained TextVectorization layer

    Returns:
        Embedding layer that converts text to embeddings
    """
    # Create embedding layer with the same vocabulary
    text_embedding = layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBED_DIM,
        mask_zero=True,  # Important for variable length sequences
        name="text_embedding"
    )

    return text_embedding

# ============================================
# DECODER FOR CAPTION GENERATION
# ============================================
def build_caption_decoder():
    """
    Builds transformer decoder for caption generation.

    Returns:
        Decoder model that takes encoder outputs and text inputs
    """
    # Two inputs:
    # 1. Encoder outputs: (batch, frames, embed_dim)
    # 2. Target captions (shifted right): (batch, caption_length)

    encoder_outputs = layers.Input(
        shape=(FRAMES_PER_CLIP, EMBED_DIM),
        name="encoder_outputs"
    )

    target_captions = layers.Input(
        shape=(MAX_CAPTION_LENGTH,),
        name="target_captions"
    )

    # Text embedding
    text_embeddings = layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBED_DIM,
        mask_zero=True,
        name="decoder_text_embedding"
    )(target_captions)

    # Positional encoding for text
    positions = tf.range(start=0, limit=MAX_CAPTION_LENGTH, delta=1)
    pos_embedding = layers.Embedding(
        input_dim=MAX_CAPTION_LENGTH,
        output_dim=EMBED_DIM,
        name="decoder_position_embedding"
    )
    pos_emb = pos_embedding(positions)
    pos_emb = tf.expand_dims(pos_emb, axis=0)  # Add batch dimension

    # Add positional encoding to text embeddings
    x = layers.Add(name="decoder_add_positional")([text_embeddings, pos_emb])
    x = layers.Dropout(DROPOUT_RATE)(x)

    # Transformer decoder layers
    for layer_idx in range(DEC_LAYERS):
        # Masked self-attention (causal masking for generation)
        self_attention = layers.MultiHeadAttention(
            num_heads=NUM_HEADS,
            key_dim=EMBED_DIM // NUM_HEADS,
            dropout=DROPOUT_RATE,
            name=f"decoder_self_attention_{layer_idx}"
        )(x, x, use_causal_mask=True)  # Causal mask for auto-regressive generation

        # Add & Norm
        x = layers.Add(name=f"decoder_add_self_{layer_idx}")([x, self_attention])
        x = layers.LayerNormalization(
            epsilon=1e-6,
            name=f"decoder_norm_self_{layer_idx}"
        )(x)

        # Cross-attention (attend to encoder outputs)
        cross_attention = layers.MultiHeadAttention(
            num_heads=NUM_HEADS,
            key_dim=EMBED_DIM // NUM_HEADS,
            dropout=DROPOUT_RATE,
            name=f"decoder_cross_attention_{layer_idx}"
        )(x, encoder_outputs)

        # Add & Norm
        x = layers.Add(name=f"decoder_add_cross_{layer_idx}")([x, cross_attention])
        x = layers.LayerNormalization(
            epsilon=1e-6,
            name=f"decoder_norm_cross_{layer_idx}"
        )(x)

        # Feed-forward network
        ff_output = layers.Dense(
            FF_DIM,
            activation='relu',
            name=f"decoder_ff1_{layer_idx}"
        )(x)
        ff_output = layers.Dropout(DROPOUT_RATE)(ff_output)
        ff_output = layers.Dense(
            EMBED_DIM,
            activation='linear',
            name=f"decoder_ff2_{layer_idx}"
        )(ff_output)

        # Add & Norm
        x = layers.Add(name=f"decoder_add_ff_{layer_idx}")([x, ff_output])
        x = layers.LayerNormalization(
            epsilon=1e-6,
            name=f"decoder_norm_ff_{layer_idx}"
        )(x)

    # Final projection to vocabulary
    decoder_output = layers.Dense(
        VOCAB_SIZE,
        activation='softmax',
        name="vocab_projection"
    )(x)

    return Model(
        inputs=[encoder_outputs, target_captions],
        outputs=decoder_output,
        name="caption_decoder"
    )


In [21]:

# ============================================
# COMPLETE VIDEO CAPTIONING MODEL
# ============================================
def build_video_captioning_model(encoder, decoder):
    """
    Combines encoder and decoder into full model.

    Args:
        encoder: Your video encoder model
        decoder: Caption decoder model

    Returns:
        Complete model for training
    """
    # Inputs
    video_input = layers.Input(
        shape=(FRAMES_PER_CLIP, IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS),
        name="video_input"
    )

    caption_input = layers.Input(
        shape=(MAX_CAPTION_LENGTH,),
        name="caption_input"
    )

    # Encode video
    video_features = encoder(video_input)

    # Decode to caption
    caption_output = decoder([video_features, caption_input])

    # Create model
    model = Model(
        inputs=[video_input, caption_input],
        outputs=caption_output,
        name="video_captioning_model"
    )

    return model

# ============================================
# TRAINING MODEL (TEACHER FORCING)
# ============================================
def build_training_model(vectorizer):
    """
    Builds complete training pipeline.

    Args:
        vectorizer: Your trained TextVectorization layer
    """
    # Build components
    encoder = build_video_encoder()  # Your existing encoder function
    decoder = build_caption_decoder()

    # Build complete model
    model = build_video_captioning_model(encoder, decoder)

    # Compile model
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-7
    )

    # Loss function (ignore padding tokens)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=False,
        ignore_class=0  # Ignore padding (0)
    )

    # Metrics
    metrics = [
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top_5_accuracy")
    ]

    model.compile(
        optimizer=optimizer,
        loss=loss_fn,
        metrics=metrics
    )

    return model

# ============================================
# INFERENCE MODEL (AUTO-REGRESSIVE)
# ============================================
def build_inference_model(training_model, vectorizer):
    """
    Builds inference model for generating captions.

    Args:
        training_model: Trained video captioning model
        vectorizer: TextVectorization layer

    Returns:
        Model for generating captions from video
    """
    # Extract encoder and decoder from trained model
    encoder = training_model.get_layer("video_encoder")
    decoder = training_model.get_layer("caption_decoder")

    # Inference inputs
    video_input = layers.Input(
        shape=(FRAMES_PER_CLIP, IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS),
        name="inference_video_input"
    )

    # Start token
    start_token = layers.Input(
        shape=(1,),
        name="start_token"
    )

    # Encode video
    video_features = encoder(video_input)

    # Auto-regressive generation
    generated_tokens = start_token

    # We'll generate tokens step by step
    # (This is a simplified version, in practice use a loop or custom layer)

    # For now, we'll reuse the decoder with a custom call
    # In practice, you'd create a custom inference loop

    return Model(
        inputs=[video_input, start_token],
        outputs=generated_tokens,
        name="video_captioning_inference"
    )



In [22]:
# ============================================
# TEST THE COMPLETE MODEL
# ============================================
def test_complete_model():
    """Test the complete video captioning pipeline."""

    print("=" * 70)
    print("TESTING COMPLETE VIDEO CAPTIONING MODEL")
    print("=" * 70)

    # Build encoder (reuse your function)
    encoder = build_video_encoder()

    # Build decoder
    decoder = build_caption_decoder()

    # Build complete model
    complete_model = build_video_captioning_model(encoder, decoder)

    print("\n1. COMPLETE MODEL SUMMARY:")
    print("-" * 40)
    complete_model.summary()

    print("\n2. TESTING FORWARD PASS:")
    print("-" * 40)

    # Create dummy inputs
    batch_size = 4

    # Video input
    dummy_video = tf.random.normal((
        batch_size,
        FRAMES_PER_CLIP,
        IMG_HEIGHT,
        IMG_WIDTH,
        IMG_CHANNELS
    ))

    # Caption input (indices from vocabulary)
    dummy_caption = tf.random.uniform(
        (batch_size, MAX_CAPTION_LENGTH),
        minval=0,
        maxval=VOCAB_SIZE,
        dtype=tf.int32
    )

    print(f"Video input shape: {dummy_video.shape}")
    print(f"Caption input shape: {dummy_caption.shape}")

    # Forward pass
    output = complete_model([dummy_video, dummy_caption])

    print(f"\nOutput shape: {output.shape}")
    print(f"Expected: (batch={batch_size}, seq_len={MAX_CAPTION_LENGTH}, vocab={VOCAB_SIZE})")

    print("\n3. OUTPUT ANALYSIS:")
    print("-" * 40)
    print(f"Output dtype: {output.dtype}")
    print(f"Output range: [{tf.reduce_min(output):.6f}, {tf.reduce_max(output):.6f}]")

    # Check softmax properties
    row_sums = tf.reduce_sum(output, axis=-1)
    print(f"\nSoftmax validation (should be close to 1.0):")
    print(f"Min row sum: {tf.reduce_min(row_sums):.6f}")
    print(f"Max row sum: {tf.reduce_max(row_sums):.6f}")
    print(f"Mean row sum: {tf.reduce_mean(row_sums):.6f}")

    return complete_model


# ============================================
# DATA PREPARATION UTILITIES
# ============================================
def prepare_training_data(video_paths, captions, vectorizer):
    """
    Prepare training data for the model.

    Args:
        video_paths: List of video file paths
        captions: List of caption strings
        vectorizer: Trained TextVectorization layer

    Returns:
        X_video: Video data (you'll need to load videos)
        X_text: Input captions (shifted right)
        y_text: Target captions (shifted left)
    """
    # Vectorize captions
    caption_vectors = vectorizer(captions).numpy()

    # Create shifted versions for teacher forcing
    # Input: [START, w1, w2, ..., wn]
    # Target: [w1, w2, ..., wn, END]

    # For simplicity, we'll use the same for input and target
    # In practice, you'd shift them
    X_text = caption_vectors
    y_text = caption_vectors

    # Note: You need to load and preprocess videos from video_paths
    # This is placeholder - implement based on your video loading

    return X_text, y_text


if __name__ == "__main__":
    # Test the complete model
    model = test_complete_model()

    print("\n" + "=" * 70)
    print("✓ MODEL BUILDING COMPLETE")
    print("=" * 70)
    print("\nNext steps:")
    print("1. Load and preprocess your video data")
    print("2. Prepare training data using prepare_training_data()")
    print("3. Train the model with model.fit()")
    print("4. Implement inference for caption generation")

TESTING COMPLETE VIDEO CAPTIONING MODEL

1. COMPLETE MODEL SUMMARY:
----------------------------------------


Model: "video_captioning_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ video_input         │ (None, 15, 224,   │          0 │ -                 │
│ (InputLayer)        │ 224, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ video_encoder       │ (1, 15, 256)      │  1,180,480 │ video_input[0][0] │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ caption_input       │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ caption_decoder     │ (1, 25, 612)      │  1,895,524 │ video_encoder[0]… │
│ (Functional)        │                   │            │ caption_input[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,076,004 (11.73 MB)

 Trainable params: 3,076,004 (11.73 MB)

 Non-trainable params: 0 (0.00 B)


2. TESTING FORWARD PASS:
----------------------------------------
Video input shape: (4, 15, 224, 224, 3)
Caption input shape: (4, 25)

Output shape: (4, 25, 612)
Expected: (batch=4, seq_len=25, vocab=612)

3. OUTPUT ANALYSIS:
----------------------------------------
Output dtype: <dtype: 'float32'>
Output range: [0.000056, 0.026091]

Softmax validation (should be close to 1.0):
Min row sum: 1.000000
Max row sum: 1.000000
Mean row sum: 1.000000

✓ MODEL BUILDING COMPLETE

Next steps:
1. Load and preprocess your video data
2. Prepare training data using prepare_training_data()
3. Train the model with model.fit()
4. Implement inference for caption generation


In [23]:
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# ============================================
# VIDEO PREPROCESSING
# ============================================
def extract_frames(video_path, num_frames=15, target_size=(224, 224)):
    """
    Extract fixed number of frames from a video.

    Args:
        video_path: Path to video file
        num_frames: Number of frames to extract (default: 30)
        target_size: Target frame size (height, width)

    Returns:
        frames: NumPy array of shape (num_frames, height, width, 3)
    """
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Calculate frame indices to sample
    if total_frames >= num_frames:
        indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    else:
        # If video has fewer frames, repeat last frame
        indices = list(range(total_frames)) + [total_frames - 1] * (num_frames - total_frames)

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            # Resize and convert BGR to RGB
            frame = cv2.resize(frame, target_size)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        else:
            # If frame read fails, use black frame
            frames.append(np.zeros((*target_size, 3), dtype=np.uint8))

    cap.release()

    # Normalize to [-1, 1] range
    frames = np.array(frames, dtype=np.float32)
    frames = (frames / 127.5) - 1.0

    return frames

# ============================================
# DATA PREPARATION
# ============================================
def prepare_dataset(dataset_pairs, vectorizer, clips_dir, num_frames=15):
    """
    Prepare complete dataset for training.

    Args:
        dataset_pairs: List of (video_filename, caption)
        vectorizer: Trained TextVectorization layer
        clips_dir: Directory containing video clips
        num_frames: Number of frames to extract per video

    Returns:
        X_video: Video data array
        X_text: Input captions (shifted right)
        y_text: Target captions (shifted left)
    """
    X_video = []
    X_text = []
    y_text = []

    print(f"Processing {len(dataset_pairs)} video-caption pairs...")

    for video_file, caption in tqdm(dataset_pairs):
        # 1. Load and preprocess video
        video_path = os.path.join(clips_dir, video_file)
        try:
            video_frames = extract_frames(video_path, num_frames=num_frames)
            X_video.append(video_frames)
        except Exception as e:
            print(f"Error loading {video_file}: {e}")
            continue

        # 2. Vectorize caption
        caption_vec = vectorizer([caption]).numpy()[0]  # Shape: (25,)

        # 3. Create shifted versions for teacher forcing
        # Input: [START, w1, w2, ..., wn-1]
        # Target: [w1, w2, ..., wn, END]

        # For simplicity, we'll use:
        # Input: [w1, w2, ..., wn] (no shift for now)
        # Target: [w1, w2, ..., wn] (same as input)
        # In practice, you should add START/END tokens

        X_text.append(caption_vec)
        y_text.append(caption_vec)

    # Convert to numpy arrays
    X_video = np.array(X_video)  # Shape: (N, num_frames, 224, 224, 3)
    X_text = np.array(X_text)    # Shape: (N, 25)
    y_text = np.array(y_text)    # Shape: (N, 25)

    print(f"\nDataset prepared:")
    print(f"  Videos: {X_video.shape}")
    print(f"  Input captions: {X_text.shape}")
    print(f"  Target captions: {y_text.shape}")

    return X_video, X_text, y_text

# ============================================
# TRAINING SETUP
# ============================================
def train_video_captioning_model(dataset_pairs, vectorizer, clips_dir):
    """
    Complete training pipeline.
    """
    # 1. Prepare dataset
    print("Step 1: Preparing dataset...")
    X_video, X_text, y_text = prepare_dataset(
        dataset_pairs, vectorizer, clips_dir, num_frames=FRAMES_PER_CLIP
    )

    # 2. Train/validation split
    print("\nStep 2: Splitting data...")
    X_video_train, X_video_val, X_text_train, X_text_val, y_text_train, y_text_val = train_test_split(
        X_video, X_text, y_text, test_size=0.2, random_state=42
    )

    print(f"Training samples: {len(X_video_train)}")
    print(f"Validation samples: {len(X_video_val)}")

    # 3. Build model
    print("\nStep 3: Building model...")
    model = build_training_model(vectorizer)

    # 4. Callbacks
    print("\nStep 4: Setting up training...")
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            'best_video_captioning_model.keras',
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        tf.keras.callbacks.TensorBoard(
            log_dir='./logs',
            histogram_freq=1
        )
    ]

    # 5. Train
    print("\nStep 5: Training model...")
    history = model.fit(
        [X_video_train, X_text_train],
        y_text_train,
        validation_data=([X_video_val, X_text_val], y_text_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )

    return model, history

# ============================================
# INFERENCE
# ============================================
def generate_caption(model, video_path, vectorizer, num_frames=30):
    """
    Generate caption for a single video.

    Args:
        model: Trained video captioning model
        video_path: Path to video file
        vectorizer: TextVectorization layer
        num_frames: Number of frames to extract

    Returns:
        Generated caption string
    """
    # 1. Preprocess video
    video_frames = extract_frames(video_path, num_frames=num_frames)
    video_frames = np.expand_dims(video_frames, axis=0)  # Add batch dimension

    # 2. Start with START token (assuming 1 is START token index)
    # In practice, you should add START token to your vocabulary
    start_token = np.ones((1, 1), dtype=np.int32)  # Shape: (1, 1)

    # 3. Auto-regressive generation (simplified)
    generated_ids = []

    for i in range(MAX_CAPTION_LENGTH):
        # Get predictions
        predictions = model.predict([video_frames, start_token], verbose=0)

        # Get next token (greedy sampling)
        next_token = np.argmax(predictions[0, -1, :])

        if next_token == 0:  # END token or padding
            break

        generated_ids.append(next_token)

        # Update input for next step
        start_token = np.append(start_token, [[next_token]], axis=1)

    # 4. Convert tokens to text
    vocabulary = vectorizer.get_vocabulary()
    caption_words = [vocabulary[token_id] for token_id in generated_ids]
    caption = ' '.join(caption_words)

    return caption



In [25]:
# First, recreate dataset_pairs if it's missing
import os
import json

# Your original setup
ANNOT_FILE = "youcookii_annotations_trainval.json"
CLIPS_DIR = "/content/drive/MyDrive/youcook2_clips_small"  # Update this

# Load annotations
with open(ANNOT_FILE, "r") as f:
    YC_FULL = json.load(f)
YC = YC_FULL["database"]

# Parse filename function (fixed version)
def parse_filename(fname):
    fname = fname.replace(".mp4", "")
    parts = fname.split("_")

    if len(parts) < 4:
        return None, None, None

    try:
        end = int(parts[-1])
        start = int(parts[-2])
        video_id = "_".join(parts[:-3])
        return video_id, start, end
    except (ValueError, IndexError):
        return None, None, None

# Get ground truth caption
def get_gt_caption(filename):
    video_id, start, end = parse_filename(filename)

    if video_id is None or video_id not in YC:
        return None

    for ann in YC[video_id]["annotations"]:
        seg_start, seg_end = ann["segment"]
        if seg_start == start and seg_end == end:
            return ann["sentence"]

    return None

# Get all clips
all_clips = [f for f in os.listdir(CLIPS_DIR) if f.endswith(".mp4")]
print(f"Found {len(all_clips)} clips in {CLIPS_DIR}")

# Create dataset_pairs
dataset_pairs = []
for f in all_clips[0:500]:
    gt = get_gt_caption(f)
    if gt:
        dataset_pairs.append((f, gt))

print(f"Created dataset_pairs with {len(dataset_pairs)} video-caption pairs")
print(f"Sample: {dataset_pairs[0] if dataset_pairs else 'No pairs found'}")

# Check if you need to recreate vectorizer too
if 'vectorizer' not in locals() and 'vectorizer' not in globals():
    from tensorflow.keras.layers import TextVectorization

    captions = [cap for _, cap in dataset_pairs]
    vectorizer = TextVectorization(
        max_tokens=5000,
        output_sequence_length=25,
        standardize='lower_and_strip_punctuation'
    )
    vectorizer.adapt(captions)
    VOCAB_SIZE = len(vectorizer.get_vocabulary())
    print(f"Created vectorizer with vocab size: {VOCAB_SIZE}")

Found 2395 clips in /content/drive/MyDrive/youcook2_clips_small
Created dataset_pairs with 500 video-caption pairs
Sample: ('pTjoGIvSfE8_seg0_154_181.mp4', 'drain and rinse cannellini beans and set aside')


In [26]:
import os
import json
import pickle
import numpy as np
import cv2
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
from sklearn.model_selection import train_test_split

# ============================================
# CONFIGURATION
# ============================================
CLIPS_DIR = "/content/drive/MyDrive/youcook2_clips_small"
ANNOT_FILE = "youcookii_annotations_trainval.json"
PROCESSED_DIR = "/content/drive/MyDrive/processed_data"
CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoints"

# Model constants
FRAMES_PER_CLIP = 15
IMG_SIZE = 224
MAX_CAPTION_LENGTH = 25
VOCAB_SIZE = 614
EMBED_DIM = 256
BATCH_SIZE = 16
EPOCHS = 10

# Create directories
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ============================================
# 1. LOAD DATASET (ONCE)
# ============================================
def load_dataset():
    """Load or create dataset_pairs."""
    print("Loading dataset...")

    # Load annotations
    with open(ANNOT_FILE, "r") as f:
        YC_FULL = json.load(f)
    YC = YC_FULL["database"]

    # Parse filename function
    def parse_filename(fname):
        fname = fname.replace(".mp4", "")
        parts = fname.split("_")
        if len(parts) < 4:
            return None, None, None
        try:
            end = int(parts[-1])
            start = int(parts[-2])
            video_id = "_".join(parts[:-3])
            return video_id, start, end
        except (ValueError, IndexError):
            return None, None, None

    # Get ground truth caption
    def get_gt_caption(filename):
        video_id, start, end = parse_filename(filename)
        if video_id is None or video_id not in YC:
            return None
        for ann in YC[video_id]["annotations"]:
            seg_start, seg_end = ann["segment"]
            if seg_start == start and seg_end == end:
                return ann["sentence"]
        return None

    # Get all clips
    all_clips = [f for f in os.listdir(CLIPS_DIR) if f.endswith(".mp4")]
    print(f"Found {len(all_clips)} clips")

    # Create dataset_pairs
    dataset_pairs = []
    for f in tqdm(all_clips, desc="Matching captions"):
        gt = get_gt_caption(f)
        if gt:
            dataset_pairs.append((f, gt))

    print(f"Created dataset with {len(dataset_pairs)} valid pairs")
    return dataset_pairs

# ============================================
# 2. CREATE VECTORIZER (WITH VERIFICATION)
# ============================================
def create_vectorizer(dataset_pairs):
    """Create and save text vectorizer."""
    vectorizer_path = os.path.join(PROCESSED_DIR, "vectorizer.pkl")

    # Check if vectorizer exists and is valid
    if os.path.exists(vectorizer_path):
        print("Loading existing vectorizer...")
        try:
            with open(vectorizer_path, 'rb') as f:
                vectorizer_data = pickle.load(f)
            vectorizer = TextVectorization.from_config(vectorizer_data['config'])
            vectorizer.set_weights(vectorizer_data['weights'])

            # Verify vocabulary size
            vocab_size = len(vectorizer.get_vocabulary())
            if vocab_size > 10:  # Should be ~1289
                print(f"✓ Loaded vectorizer with {vocab_size} words")
                return vectorizer
            else:
                print(f"✗ Vectorizer corrupted (vocab size: {vocab_size}), recreating...")
        except:
            print("✗ Vectorizer file corrupted, recreating...")

    # Create new vectorizer
    print("Creating new vectorizer...")
    captions = [cap for _, cap in dataset_pairs]
    vectorizer = TextVectorization(
        max_tokens=VOCAB_SIZE,
        output_sequence_length=MAX_CAPTION_LENGTH,
        standardize='lower_and_strip_punctuation'
    )
    vectorizer.adapt(captions)

    # Save
    vectorizer_data = {
        'config': vectorizer.get_config(),
        'weights': vectorizer.get_weights()
    }
    with open(vectorizer_path, 'wb') as f:
        pickle.dump(vectorizer_data, f)

    vocab_size = len(vectorizer.get_vocabulary())
    print(f"✓ Created vectorizer with {vocab_size} words")
    return vectorizer

# ============================================
# 3. VIDEO PROCESSING (SIMPLE CHECKPOINTING)
# ============================================
def extract_frames_safe(video_path):
    """Extract frames with robust error handling."""
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return None

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0:
            cap.release()
            return None

        # Sample frames
        if total_frames >= FRAMES_PER_CLIP:
            indices = np.linspace(0, total_frames - 1, FRAMES_PER_CLIP, dtype=int)
        else:
            indices = list(range(total_frames)) + [total_frames - 1] * (FRAMES_PER_CLIP - total_frames)

        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
            else:
                frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))

        cap.release()

        # Normalize
        frames = np.array(frames, dtype=np.float32)
        frames = (frames / 127.5) - 1.0
        return frames

    except Exception as e:
        print(f"Error extracting {video_path}: {str(e)[:50]}...")
        return None

def process_videos_with_checkpoint(dataset_pairs, vectorizer):
    """Process videos with progress tracking."""
    progress_file = os.path.join(PROCESSED_DIR, "progress.json")
    video_data_file = os.path.join(PROCESSED_DIR, "videos.npy")
    text_data_file = os.path.join(PROCESSED_DIR, "texts.npy")

    # Load progress if exists
    if os.path.exists(progress_file):
        with open(progress_file, 'r') as f:
            progress = json.load(f)
        processed_count = progress['processed']
        print(f"Resuming from {processed_count}/{len(dataset_pairs)} videos")
    else:
        progress = {'processed': 0, 'failed': []}
        processed_count = 0

    # Load existing data or create new
    if os.path.exists(video_data_file) and os.path.exists(text_data_file):
        video_data = np.load(video_data_file, allow_pickle=True)
        text_data = np.load(text_data_file, allow_pickle=True)
    else:
        video_data = []
        text_data = []

    # Process remaining videos
    start_idx = processed_count
    batch_size = 50

    print(f"\nProcessing videos {start_idx} to {len(dataset_pairs)}...")

    for i in tqdm(range(start_idx, len(dataset_pairs)), desc="Processing"):
        filename, caption = dataset_pairs[i]
        video_path = os.path.join(CLIPS_DIR, filename)

        try:
            # Extract frames
            frames = extract_frames_safe(video_path)
            if frames is None:
                raise ValueError("Frame extraction failed")

            # Vectorize caption
            caption_vec = vectorizer([caption]).numpy()[0]

            # Store
            video_data.append(frames)
            text_data.append(caption_vec)

            # Update progress
            progress['processed'] = i + 1

            # Save checkpoint every batch_size
            if (i - start_idx + 1) % batch_size == 0:
                np.save(video_data_file, np.array(video_data))
                np.save(text_data_file, np.array(text_data))
                with open(progress_file, 'w') as f:
                    json.dump(progress, f)
                print(f"  Checkpoint saved at video {i+1}")

        except Exception as e:
            print(f"\nFailed {filename}: {str(e)[:50]}")
            progress['failed'].append(filename)

    # Final save
    np.save(video_data_file, np.array(video_data))
    np.save(text_data_file, np.array(text_data))
    with open(progress_file, 'w') as f:
        json.dump(progress, f)

    print(f"\n✓ Processing complete!")
    print(f"  Success: {len(video_data)} videos")
    print(f"  Failed: {len(progress['failed'])} videos")

    return np.array(video_data), np.array(text_data)



In [27]:
# ============================================
# 4. MAIN EXECUTION
# ============================================
def main():
    """Main pipeline."""
    print("=" * 60)
    print("VIDEO CAPTIONING PIPELINE")
    print("=" * 60)

    # Step 1: Load dataset
    dataset_pairs = load_dataset()

    # Step 2: Create vectorizer
    vectorizer = create_vectorizer(dataset_pairs)

    # Step 3: Process videos
    video_data, text_data = process_videos_with_checkpoint(dataset_pairs, vectorizer)

    # Step 4: Split data
    indices = np.arange(len(video_data))
    train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

    X_video_train = video_data[train_idx]
    X_text_train = text_data[train_idx]
    y_train = text_data[train_idx]

    X_video_val = video_data[val_idx]
    X_text_val = text_data[val_idx]
    y_val = text_data[val_idx]

    print(f"\nData ready for training:")
    print(f"  Training: {len(X_video_train)} samples")
    print(f"  Validation: {len(X_video_val)} samples")

    # Step 5: Build model (you need to import/define build_training_model)
    # model = build_training_model(vectorizer)
    # ... rest of training code ...

    return video_data, text_data



In [1]:
# ============================================
# MAIN EXECUTION
# ============================================
if __name__ == "__main__":
    import os

    # Your data
    CLIPS_DIR = "/content/drive/MyDrive/youcook2_clips_small"

    # 1. Train the model
    print("Starting video captioning training...")
    model, history = train_video_captioning_model(
        dataset_pairs,
        vectorizer,
        CLIPS_DIR
    )

    # 2. Save final model
    print("\nSaving final model...")
    model.save("video_captioning_final_model.keras")

    # 3. Test inference
    print("\nTesting inference on a sample video...")
    if len(dataset_pairs) > 0:
        sample_video = os.path.join(CLIPS_DIR, dataset_pairs[0][0])
        if os.path.exists(sample_video):
            caption = generate_caption(model, sample_video, vectorizer)
            print(f"Generated caption: {caption}")
            print(f"True caption: {dataset_pairs[0][1]}")

    print("\nTraining completed!")

Starting video captioning training...


NameError: name 'train_video_captioning_model' is not defined

In [8]:
import os
import json
import pickle
import numpy as np
import cv2
from tqdm import tqdm
from tensorflow.keras.layers import TextVectorization

# ============================================
# CONFIGURATION
# ============================================
CLIPS_DIR = "/content/drive/MyDrive/youcook2_clips_small"
ANNOT_FILE = "youcookii_annotations_trainval.json"
SAVE_DIR = "/content/drive/MyDrive/processed_500"  # Yaha save karna hai

TOTAL_CLIPS = 500
FRAMES_PER_CLIP = 15
IMG_SIZE = 224
MAX_CAPTION_LENGTH = 25

# Create save directory
os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================
# 1. LOAD FIRST 500 CLIPS
# ============================================
print("Step 1: Loading 500 clips...")

# Load annotations
with open(ANNOT_FILE, "r") as f:
    YC_FULL = json.load(f)
YC = YC_FULL["database"]

# Parse filename
def parse_filename(fname):
    fname = fname.replace(".mp4", "")
    parts = fname.split("_")
    if len(parts) < 4:
        return None, None, None
    try:
        end = int(parts[-1])
        start = int(parts[-2])
        video_id = "_".join(parts[:-3])
        return video_id, start, end
    except:
        return None, None, None

# Get caption
def get_gt_caption(filename):
    video_id, start, end = parse_filename(filename)
    if video_id is None or video_id not in YC:
        return None
    for ann in YC[video_id]["annotations"]:
        seg_start, seg_end = ann["segment"]
        if seg_start == start and seg_end == end:
            return ann["sentence"]
    return None

# Get clips
all_clips = [f for f in os.listdir(CLIPS_DIR) if f.endswith(".mp4")]
dataset_pairs = []
count = 0

for f in tqdm(all_clips, desc="Loading clips"):
    if count >= TOTAL_CLIPS:
        break
    gt = get_gt_caption(f)
    if gt:
        dataset_pairs.append((f, gt))
        count += 1

print(f"✓ Loaded {len(dataset_pairs)} clips")

# ============================================
# 2. CREATE VECTORIZER
# ============================================
print("\nStep 2: Creating vectorizer...")
captions = [cap for _, cap in dataset_pairs]

vectorizer = TextVectorization(
    max_tokens=3000,
    output_sequence_length=MAX_CAPTION_LENGTH,
    standardize='lower_and_strip_punctuation',
    output_mode='int'
)
vectorizer.adapt(captions)

# Save vectorizer
vectorizer_path = os.path.join(SAVE_DIR, "vectorizer_500.pkl")
with open(vectorizer_path, 'wb') as f:
    pickle.dump({
        'config': vectorizer.get_config(),
        'weights': vectorizer.get_weights()
    }, f)

print(f"✓ Vectorizer saved: {vectorizer_path}")
print(f"Vocabulary size: {len(vectorizer.get_vocabulary())}")

# ============================================
# 3. PROCESS VIDEOS (MAIN STEP)
# ============================================
print(f"\nStep 3: Processing {len(dataset_pairs)} videos...")

video_data = []
text_data = []
failed_files = []

for i, (filename, caption) in enumerate(tqdm(dataset_pairs, desc="Processing videos")):
    try:
        # 3.1 Load and process video
        video_path = os.path.join(CLIPS_DIR, filename)
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            raise ValueError(f"Cannot open {filename}")

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # Sample frames
        if total_frames >= FRAMES_PER_CLIP:
            indices = np.linspace(0, total_frames-1, FRAMES_PER_CLIP, dtype=int)
        else:
            indices = list(range(total_frames)) + [total_frames-1] * (FRAMES_PER_CLIP - total_frames)

        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
            else:
                frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))

        cap.release()

        # Normalize to [-1, 1]
        frames = np.array(frames, dtype=np.float32)
        frames = (frames / 127.5) - 1.0

        # 3.2 Process caption
        caption_vec = vectorizer([caption]).numpy()[0]

        # 3.3 Save
        video_data.append(frames)
        text_data.append(caption_vec)

        # 3.4 Save checkpoint every 50 videos
        if (i + 1) % 50 == 0:
            print(f"  Checkpoint at video {i+1}")

    except Exception as e:
        print(f"\nFailed {filename}: {str(e)[:50]}")
        failed_files.append(filename)
        # Add zeros as placeholder
        video_data.append(np.zeros((FRAMES_PER_CLIP, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32))
        text_data.append(np.zeros((MAX_CAPTION_LENGTH,), dtype=np.int32))

# ============================================
# 4. SAVE FINAL DATA
# ============================================
print("\nStep 4: Saving final data...")

# Convert to numpy arrays
video_data_np = np.array(video_data, dtype=np.float32)
text_data_np = np.array(text_data, dtype=np.int32)

# Save to drive
video_path = os.path.join(SAVE_DIR, "video_data_500.npy")
text_path = os.path.join(SAVE_DIR, "text_data_500.npy")

np.save(video_path, video_data_np)
np.save(text_path, text_data_np)

# Save metadata
metadata = {
    'num_videos': len(dataset_pairs),
    'frames_per_clip': FRAMES_PER_CLIP,
    'image_size': IMG_SIZE,
    'max_caption_length': MAX_CAPTION_LENGTH,
    'failed_files': failed_files,
    'video_shape': video_data_np.shape,
    'text_shape': text_data_np.shape,
    'data_type': {
        'video': str(video_data_np.dtype),
        'text': str(text_data_np.dtype)
    }
}

with open(os.path.join(SAVE_DIR, "metadata_500.json"), 'w') as f:
    json.dump(metadata, f, indent=2)

# ============================================
# 5. FINAL SUMMARY
# ============================================
print("\n" + "="*60)
print("PROCESSING COMPLETE!")
print("="*60)
print(f"\nSaved to: {SAVE_DIR}")
print(f"\nFiles created:")
print(f"1. {video_path}")
print(f"   Size: {video_data_np.shape}")
print(f"   Memory: {video_data_np.nbytes / 1024**3:.2f} GB")
print(f"\n2. {text_path}")
print(f"   Size: {text_data_np.shape}")
print(f"\n3. {vectorizer_path}")
print(f"\n4. {SAVE_DIR}/metadata_500.json")
print(f"\nFailed files: {len(failed_files)}")
if failed_files:
    print("Sample failed:", failed_files[:3])

print("\n" + "="*60)
print("NEXT STEP: Download these files and train on GPU machine")
print("="*60)

Step 1: Loading 500 clips...


Loading clips:  21%|██        | 500/2395 [00:00<00:00, 164961.22it/s]


✓ Loaded 500 clips

Step 2: Creating vectorizer...
✓ Vectorizer saved: /content/drive/MyDrive/processed_500/vectorizer_500.pkl
Vocabulary size: 612

Step 3: Processing 500 videos...


Processing videos:  10%|█         | 50/500 [01:34<04:44,  1.58it/s]

  Checkpoint at video 50


...

Processing videos:  90%|█████████ | 450/500 [04:25<00:21,  2.30it/s]

  Checkpoint at video 450


Processing videos: 100%|██████████| 500/500 [04:51<00:00,  1.71it/s]

  Checkpoint at video 500

Step 4: Saving final data...



PROCESSING COMPLETE!

Saved to: /content/drive/MyDrive/processed_500

Files created:
1. /content/drive/MyDrive/processed_500/video_data_500.npy
   Size: (500, 15, 224, 224, 3)
   Memory: 4.21 GB

2. /content/drive/MyDrive/processed_500/text_data_500.npy
   Size: (500, 25)

3. /content/drive/MyDrive/processed_500/vectorizer_500.pkl

4. /content/drive/MyDrive/processed_500/metadata_500.json

Failed files: 12
Sample failed: ['wSXkTrTvI5o_seg0_40_44.mp4', 'wSXkTrTvI5o_seg1_56_86.mp4', 'wSXkTrTvI5o_seg2_92_95.mp4']

NEXT STEP: Download these files and train on GPU machine


In [ ]:
import random

random.shuffle(dataset_pairs)

train_ratio = 0.9
N = len(dataset_pairs)
split = int(train_ratio * N)

train_pairs = dataset_pairs[:split]
test_pairs  = dataset_pairs[split:]

print("Train size:", len(train_pairs))
print("Test  size:", len(test_pairs))


Train size: 1782
Test  size: 198


In [ ]:
print(train_pairs[0:5])

[('I8ckIq2-j0k_seg7_253_260.mp4', 'remove the fish from the pot'), ('zqTXQ-YqrgQ_seg10_151_160.mp4', 'top with basil leaves'), ('G-spzGkKIHM_seg3_108_262.mp4', 'drizzle oil on top and around the sides and let cook'), ('0VBNakxmATU_seg0_1_20.mp4', 'mix mayonannaise milk pepper sauce salt pepper and vinegar'), ('Cgyi5kaU7Qc_seg9_210_223.mp4', 'mix everything up')]


In [ ]:
import cv2
import numpy as np

CLIP_DIR = "/content/drive/MyDrive/youcook2_clips_small"
FRAMES_PER_CLIP = 12
IMG_SIZE = 128 # Your choice

def load_frames_from_clip(filename, num_frames=FRAMES_PER_CLIP):
    path = f"{CLIP_DIR}/{filename}"

    cap = cv2.VideoCapture(path)
    frames = []

    while True:
        ret, f = cap.read()
        if not ret:
            break
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        frames.append(f)

    cap.release()

    if len(frames) == 0:
        return None

    # sample uniformly
    idxs = np.linspace(0, len(frames)-1, num_frames).astype(int)
    sampled = [frames[i] for i in idxs]

    # resize and normalize
    out = []
    for f in sampled:
        f = cv2.resize(f, (IMG_SIZE, IMG_SIZE))
        f = f.astype("float32") / 255.0
        out.append(f)

    return np.array(out)  # (T, H, W, 3)


In [ ]:
frames = load_frames_from_clip("I8ckIq2-j0k_seg7_253_260.mp4")
print(frames.shape)


(12, 128, 128, 3)


In [ ]:
import tensorflow as tf

def generator_train():
    for fname, cap in train_pairs:
        frames = load_frames_from_clip(fname)
        if frames is None:
            continue
        yield frames, cap

train_ds = tf.data.Dataset.from_generator(
    generator_train,
    output_signature=(
        tf.TensorSpec(shape=(FRAMES_PER_CLIP, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.string)
    )
)


In [ ]:
train_ds = train_ds.shuffle(500).batch(8).prefetch(tf.data.AUTOTUNE)


In [ ]:
def generator_test():
    for fname, cap in test_pairs:
        frames = load_frames_from_clip(fname)
        if frames is None:
            continue
        yield frames, cap

test_ds = tf.data.Dataset.from_generator(
    generator_test,
    output_signature=(
        tf.TensorSpec(shape=(FRAMES_PER_CLIP, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.string)
    )
).batch(8)


In [ ]:

from tensorflow.keras.layers import TextVectorization

captions = [cap for _, cap in dataset_pairs]

vectorizer = TextVectorization(
    max_tokens=5000,
    output_sequence_length=25,
    standardize='lower_and_strip_punctuation'
)

vectorizer.adapt(captions)

VOCAB_SIZE = len(vectorizer.get_vocabulary())
print("Vocab size:", VOCAB_SIZE)


Vocab size: 1205


In [ ]:
def preprocess(frames, caption):
    tokenized = vectorizer(caption)

    decoder_in  = tokenized[:-1]   # remove last token
    decoder_out = tokenized[1:]    # remove first token

    return (frames, decoder_in), decoder_out




In [ ]:
model.fit(train_ds, validation_data=test_ds, epochs=10)


Epoch 1/10


ValueError: Layer "functional_11" expects 2 input(s), but it received 1 input tensors. Inputs received: [<tf.Tensor 'data:0' shape=(None, 12, 128, 128, 3) dtype=float32>]

In [ ]:
import tensorflow as tf
import numpy as np

class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset_pairs, vectorizer, batch_size=32, shuffle=True, frames_per_clip=FRAMES_PER_CLIP, img_size=IMG_SIZE):
        self.dataset_pairs = dataset_pairs
        self.vectorizer = vectorizer
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.frames_per_clip = frames_per_clip
        self.img_size = img_size
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.dataset_pairs) / self.batch_size))

    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_pairs = [self.dataset_pairs[k] for k in indexes]
        X_frames, X_captions, y_captions = self.__data_generation(batch_pairs)
        return [X_frames, X_captions], y_captions

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.dataset_pairs))
        if self.shuffle == True:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_pairs):
        # Directly use the known output_sequence_length value
        sequence_length = 25 # This value was set during TextVectorization initialization

        X_frames = np.empty((self.batch_size, self.frames_per_clip, self.img_size, self.img_size, 3), dtype=np.float32)
        X_captions = np.empty((self.batch_size, sequence_length), dtype=np.int64)
        y_captions = np.empty((self.batch_size, sequence_length), dtype=np.int64)

        for i, (video_path, caption_text) in enumerate(batch_pairs):
            frames = load_video_frames(video_path)
            tokenized_caption = self.vectorizer([caption_text]).numpy().flatten()

            X_frames[i,] = frames
            # Adjust padding based on the correct sequence_length
            X_captions[i,] = np.pad(tokenized_caption[:-1], (0, sequence_length - len(tokenized_caption) + 1), 'constant', constant_values=0)
            y_captions[i,] = np.pad(tokenized_caption[1:], (0, sequence_length - len(tokenized_caption) + 1), 'constant', constant_values=0)

        return X_frames, X_captions, y_captions


# Split data into training and validation sets
TRAIN_SPLIT = 0.8
split_idx = int(len(dataset_pairs) * TRAIN_SPLIT)
train_pairs = dataset_pairs[:split_idx]
val_pairs = dataset_pairs[split_idx:]

print(f"Training samples: {len(train_pairs)}")
print(f"Validation samples: {len(val_pairs)}")

# Create data generators
BATCH_SIZE = 16
train_generator = DataGenerator(train_pairs, vectorizer, batch_size=BATCH_SIZE)
val_generator = DataGenerator(val_pairs, vectorizer, batch_size=BATCH_SIZE, shuffle=False)

# Train the model
EPOCHS = 10
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2)
    ]
)


Training samples: 1584
Validation samples: 396


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


TypeError: `output_signature` must contain objects that are subclass of `tf.TypeSpec` but found <class 'list'> which is not.

In [ ]:
#.....new setup

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import os
import random
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import TextVectorization

In [ ]:
# Constants
CLIP_DIR = "/content/drive/MyDrive/youcook2_clips_small"
FRAMES_PER_CLIP = 12
IMG_SIZE = 128

# Model hyperparameters
EMBED_DIM = 256
NUM_HEADS = 4
ENC_LAYERS = 2
DEC_LAYERS = 2
FF_DIM = 512

print("Constants set successfully")
print(f"Clip directory: {CLIP_DIR}")

Constants set successfully
Clip directory: /content/drive/MyDrive/youcook2_clips_small


In [ ]:
# Cell 3: Frame Loading Function (IMPROVED)
def load_frames_from_clip(filename):
    """Load and preprocess frames from a video clip"""
    path = f"{CLIP_DIR}/{filename}"

    if not os.path.exists(path):
        return None

    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        print(f"Warning: Could not open video file: {filename}")
        return None

    frames = []

    while True:
        ret, f = cap.read()
        if not ret:
            break
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        frames.append(f)

    cap.release()

    if len(frames) == 0:
        print(f"Warning: No frames extracted from: {filename}")
        return None

    # Sample uniformly
    idxs = np.linspace(0, len(frames)-1, FRAMES_PER_CLIP).astype(int)
    sampled = [frames[i] for i in idxs]

    # Resize and normalize
    out = []
    for f in sampled:
        f = cv2.resize(f, (IMG_SIZE, IMG_SIZE))
        f = f.astype("float32") / 255.0
        out.append(f)

    return np.array(out)  # (T, H, W, 3)

# Test the function
if dataset_pairs:
    test_filename = dataset_pairs[0][0]
    test_frames = load_frames_from_clip(test_filename)
    if test_frames is not None:
        print(f"Loaded frames shape: {test_frames.shape}")
        print("Frame loading function working correctly")
    else:
        print(f"Warning: Could not load test file: {test_filename}")
else:
    print("No dataset pairs available for testing")

NameError: name 'dataset_pairs' is not defined

In [ ]:
# Shuffle dataset pairs
random.shuffle(dataset_pairs)

train_ratio = 0.9
N = len(dataset_pairs)
split = int(train_ratio * N)

train_pairs = dataset_pairs[:split]
test_pairs = dataset_pairs[split:]

print(f"Total pairs: {N}")
print(f"Train size: {len(train_pairs)}")
print(f"Test size: {len(test_pairs)}")
print(f"Sample train pair: {train_pairs[0] if train_pairs else 'No data'}")

Total pairs: 1980
Train size: 1782
Test size: 198
Sample train pair: ('GmWb7W7m2vs_seg10_290_296.mp4', 'add the fried chicken pieces to the pan')


In [ ]:
# Get all captions
captions = [cap for _, cap in dataset_pairs]

# Create vectorizer
vectorizer = TextVectorization(
    max_tokens=5000,
    output_sequence_length=25,
    standardize='lower_and_strip_punctuation',
    pad_to_max_tokens=True
)

vectorizer.adapt(captions)
VOCAB_SIZE = len(vectorizer.get_vocabulary())

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"First 10 vocabulary items: {vectorizer.get_vocabulary()[:10]}")

# Test the vectorizer
test_caption = "mix everything together"
print(f"\nTest caption: '{test_caption}'")
print(f"Tokenized: {vectorizer([test_caption]).numpy()}")
print(f"Tokenized shape: {vectorizer([test_caption]).numpy().shape}")

Vocabulary size: 1205
First 10 vocabulary items: ['', '[UNK]', np.str_('the'), np.str_('and'), np.str_('add'), np.str_('to'), np.str_('a'), np.str_('in'), np.str_('pan'), np.str_('on')]

Test caption: 'mix everything together'
Tokenized: [[ 12 193  74   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0]]
Tokenized shape: (1, 25)


In [ ]:
def preprocess_batch(frames_batch, captions_batch):
    """Prepare batch for encoder-decoder training"""
    # Tokenize captions
    tokenized = vectorizer(captions_batch)  # Shape: (batch_size, 25)

    # Create decoder inputs (shift right) and targets (shift left)
    decoder_input = tokenized[:, :-1]   # Remove last token
    decoder_target = tokenized[:, 1:]    # Remove first token

    # Return inputs and targets
    return (frames_batch, decoder_input), decoder_target

# Test preprocessing
test_frames_batch = np.random.randn(2, FRAMES_PER_CLIP, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
test_captions_batch = tf.constant(["test caption one", "test caption two"])
(test_frames, test_decoder_in), test_decoder_out = preprocess_batch(test_frames_batch, test_captions_batch)

print("Preprocessing test:")
print(f"Frames shape: {test_frames.shape}")
print(f"Decoder input shape: {test_decoder_in.shape}")
print(f"Decoder output shape: {test_decoder_out.shape}")
print("Preprocessing function working correctly")

Preprocessing test:
Frames shape: (2, 12, 128, 128, 3)
Decoder input shape: (2, 24)
Decoder output shape: (2, 24)
Preprocessing function working correctly


In [ ]:
# Cell 7: Create tf.data Dataset (FIXED VERSION)
def create_dataset(pairs, batch_size=8, shuffle=True):
    """Create tf.data.Dataset from video-caption pairs"""
    def generator():
        for fname, caption in pairs:
            # Check if file exists
            file_path = f"{CLIP_DIR}/{fname}"
            if not os.path.exists(file_path):
                print(f"Warning: File not found: {fname}")
                continue

            frames = load_frames_from_clip(fname)
            if frames is not None:
                yield frames, caption

    # Create dataset from generator
    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(FRAMES_PER_CLIP, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.string)
        )
    )

    if shuffle:
        ds = ds.shuffle(buffer_size=1000)

    # Batch first
    ds = ds.batch(batch_size)

    # Apply preprocessing
    ds = ds.map(
        lambda frames, caps: preprocess_batch(frames, caps),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    return ds.prefetch(tf.data.AUTOTUNE)

# First, let's check which files actually exist
print("Checking dataset integrity...")
existing_pairs = []
missing_files = []

for fname, caption in dataset_pairs:
    file_path = f"{CLIP_DIR}/{fname}"
    if os.path.exists(file_path):
        existing_pairs.append((fname, caption))
    else:
        missing_files.append(fname)

print(f"Total pairs: {len(dataset_pairs)}")
print(f"Existing files: {len(existing_pairs)}")
print(f"Missing files: {len(missing_files)}")

if missing_files:
    print(f"\nFirst 5 missing files:")
    for f in missing_files[:5]:
        print(f"  - {f}")

# Update dataset pairs to only include existing files
dataset_pairs = existing_pairs

# Re-split the dataset
random.shuffle(dataset_pairs)
train_ratio = 0.9
N = len(dataset_pairs)
split = int(train_ratio * N)
train_pairs = dataset_pairs[:split]
test_pairs = dataset_pairs[split:]

print(f"\nUpdated dataset:")
print(f"Train size: {len(train_pairs)}")
print(f"Test size: {len(test_pairs)}")

# Create datasets
train_ds = create_dataset(train_pairs, batch_size=8, shuffle=True)
test_ds = create_dataset(test_pairs, batch_size=8, shuffle=False)

print("\nDatasets created successfully")
print(f"Training dataset: {train_ds}")
print(f"Test dataset: {test_ds}")

# Test dataset shapes
print("\nTesting dataset shapes:")
for (frames_input, caption_input), caption_target in train_ds.take(1):
    print(f"Frames input shape: {frames_input.shape}")      # (8, 12, 128, 128, 3)
    print(f"Caption input shape: {caption_input.shape}")    # (8, 24)
    print(f"Caption target shape: {caption_target.shape}")  # (8, 24)
    break

Datasets created successfully
Training dataset: <_PrefetchDataset element_spec=((TensorSpec(shape=(None, 12, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 24), dtype=tf.int64, name=None)), TensorSpec(shape=(None, 24), dtype=tf.int64, name=None))>
Test dataset: <_PrefetchDataset element_spec=((TensorSpec(shape=(None, 12, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 24), dtype=tf.int64, name=None)), TensorSpec(shape=(None, 24), dtype=tf.int64, name=None))>

Testing dataset shapes:
